# BruSafe · Notebook 03L — Module M2 run on measured data

*Literature variant of notebook 03. The swap point reads `data/lit_M2_partition_gels.csv` (solubility of arenin at two induction temperatures, doi 10.1186/s40643-025-00986-2, Table 1) instead of generating synthetic observations. Everything upstream of the swap point is unchanged.*

*Section numbering follows the CAPSID framework document; this notebook implements Sections 3.1–3.14.*

**iGEM Tec-Chihuahua 2026 · CAPSID framework**

This notebook is the companion to `BruSafe_02_M1_expression_lactose.ipynb`. It closes a gap that the coupled pipeline would otherwise leave open.

## The gap

M1 produces $P(t)$, the cumulative CP monomer. M3 (Zlotnick T=3 assembly) consumes assembly-competent
**dimer**. Nothing between the two converts one into the other, so the pipeline implicitly assumes that
every translated chain folds and is available for assembly. At T7 flux in BL21(DE3) this is not the
case: nascent chains partition between folding, aggregation into inclusion bodies and proteolysis, and
the split depends on the synthesis flux itself.

## What this notebook adds

$$\text{U} \xrightarrow{k_f} \text{N} \quad\text{(native)} \qquad
  \text{U} \xrightarrow{k_d} \text{degraded} \qquad
  n\,\text{U} \xrightarrow{k_a} \text{A} \quad\text{(inclusion body)}$$

Folding, aggregation and proteolysis occur on $10^0$–$10^2$ s, whereas expression occurs on
$10^3$–$10^4$ s. $U$ is therefore at quasi-steady state on the timescale of the expression ODEs, and
the partition reduces to a **closed-form function of the flux** with one shape parameter and one
temperature parameter.

## Why this is the composition layer and not another module

The partition is a **splitting** of $P$, not a modification of it. §4 shows that the identity

$$\mathrm{Sol}(t) + \mathrm{Agg}(t) + \mathrm{Cap}(t) \equiv P(t)$$

holds exactly, for any $\varphi$ and any assembly sink, provided that all three pools carry the same
loss rate $L = \mu + k_{deg}$ that M1 already applies. This identity is what makes the two-stage fit
legitimate: stage 1 (M1 on total CP) is not affected by stage 2.

## Workflow

Same structure as the M1 notebooks: derivation → closed-form verification → observation model →
bridge and conservation proof → identifiability audit → data (one swap point) → SMC calibration →
recovery check → **held-out temperature validation** → export for M3.

---
## 1 · Setup

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.optimize import brentq
from scipy.integrate import solve_ivp
from scipy.interpolate import interp1d
from scipy.special import gammaln
from scipy.stats import t as student_t

RNG = np.random.default_rng(20261013)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 9})
R_KCAL = 1.98720425e-3          # kcal / (mol K)
print("M2 · soluble partition | numpy", np.__version__)

# ---- Figure style (team palette) -------------------------------------------
# The colours are sampled from the team wordmark. Figures refer to the names
# below instead of hex codes, so the whole set can be restyled from this cell.
# SF Pro is used where installed; matplotlib substitutes another font otherwise.
CREAM, PALE, STEEL = "#F4F3EF", "#CCD8E2", "#89A1C2"
SKY,   BLUE, MID   = "#4CA3DF", "#3E77E1", "#2C5BA5"
NAVY,  INK         = "#173779", "#122047"
from matplotlib.colors import LinearSegmentedColormap
BRAND_CMAP = LinearSegmentedColormap.from_list("tecchihuahua",
                                               [NAVY, MID, BLUE, SKY])
plt.rcParams.update({
    "font.family": ["SF Pro Text", "SF Pro Display", "SF Pro", "Helvetica Neue"],
    "mathtext.fontset": "custom", "mathtext.default": "regular",
    "mathtext.rm": "SF Pro Text", "mathtext.it": "SF Pro Text:italic",
    "mathtext.bf": "SF Pro Text:bold",
    "figure.facecolor": CREAM, "axes.facecolor": CREAM, "savefig.facecolor": CREAM,
    "text.color": INK, "axes.labelcolor": INK, "axes.titlecolor": INK,
    "axes.edgecolor": INK, "xtick.color": INK, "ytick.color": INK,
    "grid.color": STEEL, "grid.alpha": 0.35,
    "legend.facecolor": CREAM, "legend.edgecolor": STEEL, "legend.framealpha": 0.9,
    "axes.prop_cycle": plt.cycler(color=[NAVY, BLUE, SKY, STEEL, MID, PALE]),
})


---
## 2 · The partition model

### 2.1 · Derivation

A nascent chain $U$ has three possible fates. On the expression timescale $U$ is at quasi-steady state:

$$s \;=\; k_1 U \;+\; n\,k_a U^{n}, \qquad k_1 = k_f + k_d$$

Define $\varphi = k_1 U / s$, the fraction that escapes aggregation, and $\varepsilon = k_f/k_1$, the
fraction of the escapers that fold instead of being degraded. The three fluxes are then **exact**:

| Fate | Flux | Observable |
|---|---|---|
| native, assembly-competent | $\varepsilon\,\varphi\,s$ | soluble lane |
| proteolysed | $(1-\varepsilon)\,\varphi\,s$ | neither lane |
| aggregated | $(1-\varphi)\,s$ | pellet lane |

They sum to $s$ by construction, so no mass is created or lost in the split.

### 2.2 · Closed form for $n = 2$

Setting $\varphi = 1/2$ at $s = s_{50}$ fixes $k_a$, and the quadratic then solves to

$$\boxed{\;\varphi(s) \;=\; \frac{2\left(\sqrt{1+\sigma}-1\right)}{\sigma}, \qquad \sigma = \frac{8s}{s_{50}}\;}$$

with the limits $\varphi \to 1 - s/2s_{50}$ at low flux and $\varphi \to \sqrt{s_{50}/2s}$ at high flux.
The result has one dimensionless group and one parameter.

**The $s^{-1/2}$ decay at high flux is a firm prediction of the model.** If the measured soluble
fraction falls faster than this, aggregation is nucleated ($n > 2$) or chaperones are being
exhausted. §5 fits $n$ as a phenomenological sharpness parameter and does not model chaperone
titration explicitly; the reasoning is given there.

### 2.3 · Temperature

Because $s_{50} = k_f^2/k_a$, $\ln s_{50}$ is a difference of Arrhenius terms:

$$\ln s_{50}(T) = \ln s_{50}^{ref} + \frac{\Delta E}{R}\left(\frac1T - \frac1{T_{ref}}\right),
\qquad \Delta E = E_{a,agg} - 2E_{a,fold}$$

$\Delta E > 0$ explains why induction at lower temperature improves solubility. The closure adds a
single parameter, which allows a falsification test: fitting at 37 °C and 25 °C and predicting
18 °C with no free parameters (§10). In this notebook the calibration uses the two measured
temperatures, 37 and 30 °C, and the prediction is made at 25 and 18 °C.

In [ ]:
def phi_shape(s, s50, n=2.0):
    '''Fraction of synthesis flux escaping aggregation.

    Solves  x = u + 2^(n-1) u^n   with x = s/s50, u = U/s50; returns u/x.
    n = 2 uses the closed form.'''
    s = np.asarray(s, float); x = s / s50
    if abs(n - 2.0) < 1e-12:
        with np.errstate(divide="ignore", invalid="ignore"):
            out = (np.sqrt(1.0 + 8.0*x) - 1.0) / (4.0*x)
        out = np.where(x < 1e-9, 1.0 - x, out)
        return np.clip(out, 1e-12, 1.0)
    # Vectorised bisection. x is 2-D (dose x time) inside the bridge, so a
    # per-element scalar root-find is both wrong (iterating a 2-D array yields
    # ROWS) and far too slow inside an SMC. f(u) = u + c u^n - x is strictly
    # increasing with f(0) < 0 <= f(x), so [0, x] always brackets the root.
    c  = 2.0**(n - 1.0)
    xs = np.maximum(x, 1e-300)
    lo = np.zeros_like(xs); hi = xs.copy()
    for _ in range(60):
        mid = 0.5*(lo + hi)
        neg = (mid + c*mid**n - xs) < 0.0
        lo  = np.where(neg, mid, lo); hi = np.where(neg, hi, mid)
    out = np.where(xs < 1e-12, 1.0 - c*xs**(n-1.0), 0.5*(lo+hi)/xs)
    return np.clip(out, 1e-12, 1.0)


def s50_at_T(s50_ref, dE_over_R, T_K, T_ref_K=310.15):
    return s50_ref * np.exp(dE_over_R * (1.0/np.asarray(T_K, float) - 1.0/T_ref_K))


def partition_fluxes(s, s50, eps, n=2.0):
    ph = phi_shape(s, s50, n)
    return eps*ph*s, (1.0 - eps)*ph*s, (1.0 - ph)*s

### 2.4 · Verification — closed form, asymptotes, and mass balance

In [ ]:
# (a) closed form must equal the numerical root-find at n = 2
xs = np.logspace(-4, 4, 40)
cf = phi_shape(xs, 1.0, 2.0)
nm = phi_shape(xs, 1.0, 2.0 + 1e-9)         # forces the brentq branch
print(f"(a) closed form vs numerical, n=2 : max |diff| = {np.abs(cf-nm).max():.2e}")

# (b) asymptotes
lo = phi_shape(1e-4, 1.0); hi = phi_shape(1e4, 1.0)
print(f"(b) low-flux  phi -> 1 - s/2s50 : {float(lo):.6f} vs {1-1e-4/2:.6f}")
print(f"    high-flux phi -> sqrt(s50/2s): {float(hi):.6f} vs {np.sqrt(1/(2*1e4)):.6f}")
print(f"    phi(s = s50)                 : {float(phi_shape(1.0,1.0)):.6f}  (must be 0.5)")

# (c) the three fluxes must sum to s exactly
s_test = np.logspace(-2, 3, 7)
Jn, Jd, Ja = partition_fluxes(s_test, 50.0, 0.9)
print(f"(c) mass balance sum(J) - s      : max |resid| = {np.abs(Jn+Jd+Ja-s_test).max():.2e}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9.5, 3.4))
sg = np.logspace(-1, 3.5, 400)
for T_C, c in zip((37, 25, 18), [NAVY, BLUE, SKY]):
    ax[0].plot(sg, phi_shape(sg, s50_at_T(60.0, 12000.0, 273.15+T_C)),
               color=c, label=f"{T_C} $^\\circ$C")
ax[0].plot(sg, np.sqrt(60.0/(2*sg)), ls=":", color=INK, lw=1, label=r"$\sqrt{s_{50}/2s}$")
ax[0].set(xscale="log", ylim=(0, 1.05), xlabel="flux $s$ (nM s$^{-1}$)",
          ylabel=r"$\varphi$", title="Partition function")
ax[0].legend(fontsize=7)

for n_, c in zip((2.0, 3.0, 5.0), [SKY, BLUE, NAVY]):
    ax[1].plot(sg, phi_shape(sg, 60.0, n_), color=c, label=f"n = {n_:.0f}")
ax[1].set(xscale="log", yscale="log", xlabel="flux $s$ (nM s$^{-1}$)", ylabel=r"$\varphi$",
          title="Aggregation")
ax[1].legend(fontsize=7)
fig.tight_layout(); plt.show()
print("Right panel: the high-flux slope IS the aggregation order. That is what the")
print("titration measures, and why a wide flux range matters more than many doses.")

---
## 3 · Observation model: what densitometry measures

The likelihood below is valid only if three conditions hold. All three are protocol decisions and
not modelling choices.

**(a) Loading must be OD-matched.** Densitometry is compositional and gain-dependent, so absolute
band volumes are not comparable between gels. The soluble *fraction* is comparable, but only if the
soluble and pellet samples are loaded at matched culture-OD equivalents, with the whole pellet
resuspended to the same volume as the cleared lysate. If the protocol differed, the ratio is not
$\varphi$ and this section does not apply.

**(b) Capsid CP is in the soluble lane.** Assembled VLPs remain in the supernatant after a clearing
spin of $\le 20{,}000\,g$. At $100{,}000\,g$ a 27 nm particle pellets, and the soluble lane no
longer has the meaning assumed by the model. §4.4 quantifies the consequence.

**(c) Degraded CP is in neither lane.** The observed fraction is therefore a ratio of *survivors*,
not of synthesis:

$$p_{gel} = \frac{\mathrm{Sol} + \mathrm{Cap}}{\mathrm{Sol} + \mathrm{Cap} + \mathrm{Agg}}$$

At steady flux this reduces to $\varepsilon\varphi / (\varepsilon\varphi + 1 - \varphi)$, which is
**not** $\varphi$. Proteolysis biases the observed fraction downward and is confounded with
aggregation unless $\varepsilon$ is fixed. §4.5 resolves this by setting $\varepsilon = 1$.

The noise is modelled with a Beta likelihood whose concentration $\nu$ is estimated and not assumed.

In [ ]:
from scipy.special import betainc, betaincc as betainc_sf
def beta_logpdf(y, p, nu):
    a = np.clip(p*nu, 1e-6, None); b = np.clip((1-p)*nu, 1e-6, None)
    y = np.clip(y, 1e-6, 1-1e-6)
    return (gammaln(a+b) - gammaln(a) - gammaln(b)
            + (a-1)*np.log(y) + (b-1)*np.log1p(-y))

# ---- saturated bands -------------------------------------------------------
# A gel cannot report a soluble fraction of exactly 1: it reports "no pellet band
# above the detection limit". The beta density has no mass at the boundary, so the
# earlier version clipped y to 1e-6 and evaluated the density there -- which for
# p -> 1 is a huge, sharply parameter-dependent number. Simulation-based
# calibration showed the consequence: 14% of synthetic observations landed on that
# clip and the posterior came out ~1.8x too narrow for every parameter.
# Treat boundary observations as CENSORED instead, which is what they are.
BAND_DL = 0.01        # smallest band intensity resolvable as a fraction of the lane

def censor(y, dl=BAND_DL):
    """Round an observed fraction to the detection limit, as a gel reader would."""
    y = np.asarray(y, float)
    return np.where(y >= 1-dl, 1.0, np.where(y <= dl, 0.0, y))

def beta_loglik(y, p, nu, dl=BAND_DL):
    """Beta log-likelihood with the two boundary outcomes treated as censored:
    y = 1 contributes P(Y > 1-dl) and y = 0 contributes P(Y < dl)."""
    y = np.asarray(y, float)
    a = np.clip(p*nu, 1e-6, None); b = np.clip((1-p)*nu, 1e-6, None)
    out = np.empty_like(y)
    hi = y >= 1.0; lo = y <= 0.0; mid = ~(hi | lo)
    if np.any(mid):
        out[mid] = beta_logpdf(y[mid], np.asarray(p)[mid] if np.ndim(p) else p,
                               nu[mid] if np.ndim(nu) else nu)
    if np.any(hi):
        out[hi] = np.log(np.clip(betainc_sf(a[hi], b[hi], 1-dl), 1e-300, None))
    if np.any(lo):
        out[lo] = np.log(np.clip(betainc(a[lo], b[lo], dl), 1e-300, None))
    return out

def steady_gel_fraction(s, s50, eps, n=2.0):
    ph = phi_shape(s, s50, n)
    return eps*ph / (eps*ph + (1.0 - ph))

s_demo = np.logspace(0, 3, 200)
plt.figure(figsize=(4.6, 3.2))
for e, c in zip((1.0, 0.9, 0.7), [SKY, BLUE, NAVY]):
    plt.plot(s_demo, steady_gel_fraction(s_demo, 60.0, e), color=c, label=f"$\\varepsilon$ = {e}")
plt.plot(s_demo, phi_shape(s_demo, 60.0), ls="--", color=INK, lw=1, label=r"$\varphi$ itself")
plt.xscale("log"); plt.ylim(0, 1.05); plt.legend(fontsize=7)
plt.xlabel("flux (nM s$^{-1}$)"); plt.ylabel("gel soluble fraction")
plt.title("Observable $\\neq$ $\\varphi$ when proteolysis is active"); plt.tight_layout(); plt.show()

---
## 4 · The bridge from M1

### 4.1 · What crosses the boundary

| From M1 | Symbol | Units |
|---|---|---|
| gross synthesis flux | $s = G\,\tilde m\,\psi$ | µM min⁻¹ |
| shared loss rate | $L = \mu(t) + k_{deg}$ | min⁻¹ |
| total CP pool (for the check) | $P$ | µM |

The flux is converted to nM s⁻¹ (**×1000/60**) only where $\varphi$ is evaluated, because $s_{50}$ is
defined on that scale. The quantity passed to M3 is **Sol/2**, in µM dimer: MS2 CP is an obligate
dimer, and the assembly unit is the dimer and not the monomer.

> **Note.** The partition acts on the flux of nascent chains, not on the accumulated pool $P(t)$.
> Applying a partition function to $P$ is the specific error that this module is designed to avoid.

### 4.2 · The conservation identity

$$\frac{d\mathrm{Sol}}{dt} = \varphi s - \text{sink} - L\,\mathrm{Sol}, \qquad
\frac{d\mathrm{Agg}}{dt} = (1-\varphi)s - L\,\mathrm{Agg}, \qquad
\frac{d\mathrm{Cap}}{dt} = \text{sink} - L\,\mathrm{Cap}$$

Summing the three equations gives $d(\mathrm{Sol+Agg+Cap})/dt = s - L(\mathrm{Sol+Agg+Cap})$ from
zero initial conditions, which is exactly the M1 equation for $P$. This is verified numerically below.

In [ ]:
# ---- M1 forward model, reimplemented in plain scipy (no numba / pymc) -----
# Verified against the calibration notebook: P(18h) = 2729 uM on the 5500 uM arm,
# which maps to 178 mg/L through the notebook's yield conversion, matching the lactose calibration.
P_REF, PHI_CYT, GDCW_PER_OD, M_CP = 18000.0, 1.8e-3, 0.33, 13.7
T_END, GLC0 = 1080.0, 2500.0
DOSES = np.array([500., 1500., 5500., 15000.])
FIXED = dict(Xmax=8.0, q_Glc=20.0, K_upt=100.0, K_ie=500.0,
             K_Y=400.0, f_tg=0.35, K_ML=1500.0, K_MA=1200.0)
TRUE = dict(mu_max=0.021, phi_crit=0.25, k_deg=5e-4, K_cat=60.0, V_Y=320.0,
            k_cat=3000.0, K_A=150.0, h=2.0, eps=0.005, k_lac=0.05,
            k_slow=0.07, k_fast=0.30, G=95.0, **FIXED)
PORDER = ["mu_max","Xmax","phi_crit","k_deg","q_Glc","K_upt","K_ie","K_cat","V_Y",
          "K_Y","k_cat","f_tg","K_ML","K_MA","K_A","h","eps","k_lac","k_slow","k_fast","G"]
FIT_M04 = ["mu_max","phi_crit","k_deg","K_cat","V_Y","k_cat","K_A","h","eps",
           "k_lac","k_slow","k_fast","G"]

def _rhs_m04(t, Yv, p, n):
    (mu_max,Xmax,phi_crit,k_deg,q_Glc,K_upt,K_ie,K_cat,V_Y,K_Y,k_cat,f_tg,
     K_ML,K_MA,K_A,h,eps,k_lac,k_R,k_m,G) = p
    X,Glc,Le,Li,Al,B,Yp,R,m,P = [np.maximum(v,0.0) for v in Yv.reshape(10,n)]
    psi = np.maximum(1.0-(P/P_REF)/phi_crit, 0.0)
    mu  = mu_max*np.maximum(1.0-X/Xmax, 0.0)*psi
    Cc, xie = 1.0/(1.0+Glc/K_cat), 1.0/(1.0+Glc/K_ie)
    v_up = V_Y*Yp*(Le/(K_Y+Le))*xie
    sL   = k_cat*B*(Li/(K_ML+Li)); v_tg, v_hyd = f_tg*sL, (1-f_tg)*sL
    v_ah = k_cat*B*(Al/(K_MA+Al))
    f    = eps + (1-eps)*Al**h/(K_A**h + Al**h)
    return np.concatenate([mu*X, -q_Glc*X*(Glc/(K_upt+Glc)),
        -v_up*PHI_CYT*GDCW_PER_OD*X, v_up-v_hyd-v_tg-mu*Li, v_tg-v_ah-mu*Al,
        k_lac*(Cc*f-B), k_lac*(Cc*f-Yp), k_R*(f-R), k_m*(R-m),
        G*m*psi-(mu+k_deg)*P])

def simulate_m04(d, Le0, t_eval, X0=0.05, Glc0=GLC0):
    p = np.array([d[k] for k in PORDER], float); n = len(Le0)
    Y0 = np.concatenate([np.full(n,X0), np.full(n,Glc0), np.asarray(Le0,float)]+[np.zeros(n)]*7)
    s = solve_ivp(_rhs_m04,(0.,t_eval[-1]),Y0,t_eval=t_eval,args=(p,n),
                  method="LSODA",rtol=1e-7,atol=1e-10)
    Y = s.y.reshape(10,n,-1)
    out = dict(zip(["X","Glc","Le","Li","Al","B","Yp","R","m","P"], Y)); out["t"] = s.t
    psi = np.maximum(1.0-(out["P"]/P_REF)/d["phi_crit"], 0.0)
    out["psi"] = psi
    out["flux_uM_min"] = d["G"]*out["m"]*psi
    out["mu"] = d["mu_max"]*np.maximum(1.0-out["X"]/d["Xmax"],0.0)*psi
    return out

_t = np.linspace(0, T_END, 1201)
_s = simulate_m04(TRUE, DOSES, _t)
print("M1 reproduction check")
print(f"  P(18h), 5500 uM arm : {_s['P'][2,-1]:.1f} uM")
Y18 = _s['P'][2,-1]*1e-6*M_CP*1e3*PHI_CYT*(_s['X'][2,-1]*GDCW_PER_OD)*1e3
print(f"  -> yield            : {Y18:.1f} mg/L   (lactose calibration reports 178)")
print(f"  mu at 18 h          : {_s['mu'][:,-1]}   <- growth fully arrested")

In [ ]:
FLUX_UM_MIN_TO_NM_S = 1000.0/60.0

def bridge(theta, doses, s50_ref, dE_over_R, T_K=310.15, n=2.0, t_end=T_END,
           npts=1201, assembly_sink=None, capsid_in_soluble=True, k_deg_agg=None):
    '''Run M1, then split its flux into soluble / aggregate / capsid.'''
    t   = np.linspace(0.0, t_end, npts)
    sim = simulate_m04(theta, doses, t)
    L   = sim["mu"] + theta["k_deg"]
    s   = sim["flux_uM_min"]
    ph  = phi_shape(s*FLUX_UM_MIN_TO_NM_S, s50_at_T(s50_ref, dE_over_R, T_K), n)
    nd  = len(doses)
    Lagg = L if k_deg_agg is None else sim["mu"] + k_deg_agg
    I = lambda A: [interp1d(t, A[i], kind="cubic", assume_sorted=True) for i in range(nd)]
    f_s, f_L, f_p, f_La = I(s), I(L), I(ph), I(Lagg)

    def rhs(tt, y):
        Sol, Agg, Cap = y.reshape(3, nd)
        si = np.array([f(tt) for f in f_s]); Li = np.array([f(tt) for f in f_L])
        La = np.array([f(tt) for f in f_La])
        pi_ = np.clip(np.array([f(tt) for f in f_p]), 0.0, 1.0)
        sink = np.zeros(nd) if assembly_sink is None else assembly_sink(Sol, tt)
        return np.concatenate([pi_*si - sink - Li*Sol,
                               (1-pi_)*si - La*Agg,
                               sink - Li*Cap])

    sol = solve_ivp(rhs,(0.,t_end),np.zeros(3*nd),t_eval=t,method="LSODA",rtol=1e-9,atol=1e-12)
    Sol, Agg, Cap = sol.y.reshape(3, nd, -1)
    lane_s = Sol + Cap if capsid_in_soluble else Sol
    lane_p = Agg if capsid_in_soluble else Agg + Cap
    return {"t": t, "P_M1": sim["P"], "flux_nM_s": s*FLUX_UM_MIN_TO_NM_S, "phi": ph,
            "Sol": Sol, "Agg": Agg, "Cap": Cap, "total": Sol+Agg+Cap,
            "gel_soluble_fraction": lane_s/np.maximum(lane_s+lane_p, 1e-30),
            "dimer_uM": Sol/2.0, "sim": sim}

### 4.3 · The fast path: quadrature instead of integration

With `sink = 0` the split is **linear** in the pools, so it has a closed solution and needs no ODE
solver. Writing $E(t) = \exp\left(-\int_0^t L\right)$,

$$\mathrm{Sol}(t)+\mathrm{Cap}(t) = E(t)\int_0^t \frac{\varphi s}{E}\,d\tau, \qquad
\mathrm{Agg}(t) = E(t)\int_0^t \frac{(1-\varphi)s}{E}\,d\tau$$

$\mathrm{Sol}+\mathrm{Cap}$ obeys this expression regardless of the sink, because the sink only moves
CP between those two pools (§4.6), and that sum is exactly the observable. The M1 solution does not
depend on the partition parameters, so it is computed once and cached. Together these make the
likelihood about 1000 times cheaper, which is what makes the SMC in §7 tractable.

In [ ]:
def partition_quadrature(sim, theta, s50_ref, dE_over_R, T_K=310.15, n=2.0):
    '''Closed-form split. Returns (soluble lane, pellet lane) in uM.'''
    t  = sim["t"]
    L  = sim["mu"] + theta["k_deg"]
    s  = sim["flux_uM_min"]
    ph = phi_shape(s*FLUX_UM_MIN_TO_NM_S, s50_at_T(s50_ref, dE_over_R, T_K), n)
    cumtrap = lambda A: np.concatenate(
        [np.zeros((A.shape[0],1)), np.cumsum(0.5*(A[:,1:]+A[:,:-1])*np.diff(t), axis=1)], axis=1)
    E = np.exp(-cumtrap(L))
    acc = lambda g: E*cumtrap(g/E)
    return acc(ph*s), acc((1.0-ph)*s)

_b = bridge(TRUE, DOSES, 50.0, 12000.0)
_ls, _lp = partition_quadrature(_b["sim"], TRUE, 50.0, 12000.0)
_fq = _ls/np.maximum(_ls+_lp, 1e-30)
print(f"quadrature vs ODE, gel soluble fraction : max |diff| = "
      f"{np.abs(_fq - _b['gel_soluble_fraction']).max():.2e}")
print(f"quadrature vs ODE, total pool           : max |diff| = "
      f"{np.abs((_ls+_lp) - _b['total']).max():.2e} uM")

### 4.4 · Conservation and assembly-invariance tests

In [ ]:
def demo_sink(Sol, t):                      # arbitrary nonlinear assembly sink
    S = np.maximum(Sol, 0.0)
    return 3e-4*S**2/(1.0 + 1e-3*S)

rows = []
for lab, kw in [("no sink", {}), ("with assembly sink", {"assembly_sink": demo_sink})]:
    b = bridge(TRUE, DOSES, 50.0, 12000.0, **kw)
    ad = np.abs(b["total"] - b["P_M1"])
    rows.append({"case": lab, "max|dev|/max(P)": f"{ad.max()/b['P_M1'].max():.2e}",
                 "dev at 18 h (uM)": f"{np.abs(b['total'][:,-1]-b['P_M1'][:,-1]).max():.2e}"})
b_bad = bridge(TRUE, DOSES, 50.0, 12000.0, k_deg_agg=0.0)
rows.append({"case": "k_deg_agg = 0 (IBs protease-resistant)",
             "max|dev|/max(P)": f"{(np.abs(b_bad['total']-b_bad['P_M1']).max()/b_bad['P_M1'].max()):.2%}",
             "dev at 18 h (uM)": f"{np.abs(b_bad['total'][:,-1]-b_bad['P_M1'][:,-1]).max():.1f}"})
display(pd.DataFrame(rows))

b0 = bridge(TRUE, DOSES, 50.0, 12000.0)
b1 = bridge(TRUE, DOSES, 50.0, 12000.0, assembly_sink=demo_sink)
b2 = bridge(TRUE, DOSES, 50.0, 12000.0, assembly_sink=demo_sink, capsid_in_soluble=False)
print(f"\nmax change in gel soluble fraction when assembly is switched ON : "
      f"{np.abs(b0['gel_soluble_fraction']-b1['gel_soluble_fraction']).max():.2e}")
print(f"same, but with capsids pelleting (hard spin)                    : "
      f"{np.abs(b0['gel_soluble_fraction']-b2['gel_soluble_fraction']).max():.2f}")

**Reading the table.** The identity holds to solver tolerance for any $\varphi$ and any sink. Stage 1
is therefore unaffected by stage 2, which shows that the two-stage fit is legitimate and not merely
convenient. Giving inclusion bodies their own degradation rate $k_{deg,agg}$ breaks the identity by
about 13%, so that variant is useful only as a sensitivity analysis; adopting it would require
refitting M1.

**Reading the two numbers below the table.** The soluble lane holds $\mathrm{Sol}+\mathrm{Cap}$, and
assembly only moves CP between the two, so the gel fraction is invariant to the assembly sink to
$10^{-9}$. $s_{50}$ can therefore be fitted with sink = 0, and M3 switched on afterwards, without
biasing anything. The invariance fails if VLPs pellet (a shift of 0.50), in which case stages 2 and 3
must be fitted jointly. The spin speed should therefore be recorded.

### 4.5 · Four preconditions

1. **$\varepsilon = 1$ in M2.** The $k_{deg}$ of M1 already carries proteolysis, and the
   $(1-\varepsilon)$ branch represents the same process. Fitting both makes them trade off, since
   nothing at 18 h can separate them. One is fixed, and the choice is stated.
2. **The stage-1 band must come from whole-cell lysate.** If it came from cleared supernatant, $P$
   is already partitioned, $G$ and $k_{deg}$ have absorbed $\varphi$, and the bridge double-counts.
   This would show up as dose-ordered residuals in the stage-1 posterior predictive.
3. **Capsids are in the soluble lane** (§4.4).
4. **$\psi$ remains a function of total $P$**, not of soluble $P$, because this is what stage 1
   calibrated.

### 4.6 · Why the stages can be split

Stage 1 fits the total band volume; stage 2 fits the soluble fraction from the same two lanes. These
are close to orthogonal statistics: the shared gel gain $\alpha_g$ dominates the total and cancels
exactly in the ratio. The same gel therefore informs both stages without materially re-using the
data. This is a Bayesian **cut**, not a joint posterior. Fitting jointly would be defensible;
fitting stage 2 and then allowing it to revise $G$ would not.

---
## 5 · Identifiability audit

Two questions are answered by computation and not by assertion: which experimental axis moves the
observable, and which parameters trade off against each other.

In [ ]:
print("Which axis moves the observable?\n")
print(f"{'s50 (nM/s)':>11} | soluble fraction at 18 h, per lactose arm | spread")
for s50 in (20., 50., 120., 300.):
    f18 = bridge(TRUE, DOSES, s50, 12000.0)["gel_soluble_fraction"][:, -1]
    print(f"{s50:>11.0f} | " + "  ".join(f"{v:.3f}" for v in f18) + f" | {np.ptp(f18):.3f}")

print(f"\n{'T (C)':>11} | soluble fraction at 18 h, per lactose arm | spread")
for T_C in (37., 25., 18.):
    f18 = bridge(TRUE, DOSES, 50.0, 12000.0, T_K=273.15+T_C)["gel_soluble_fraction"][:, -1]
    print(f"{T_C:>11.0f} | " + "  ".join(f"{v:.3f}" for v in f18) + f" | {np.ptp(f18):.3f}")

pk = bridge(TRUE, DOSES, 50.0, 12000.0)["flux_nM_s"].max(axis=1)
print(f"\npeak flux across the dose panel: {pk.min():.0f} - {pk.max():.0f} nM/s "
      f"(factor {pk.max()/pk.min():.1f})")

**Dose is a usable but weak lever; temperature is the strong one.** (The figures in this paragraph refer to the four-dose lactose panel of notebook 03; the published measurements used here contain a single induction condition.) Peak flux spans only ~2.6×
across the lactose panel, because LacY uptake is already saturated at every dose ($K_Y = 400$ µM,
and the lowest arm is 500 µM). The resulting spread in soluble fraction is ~0.10–0.12, about twice
the per-point densitometry noise of 0.04–0.06, which is marginal but real. Temperature moves the
same quantity by ~0.35 between 37 and 18 °C.

> **Caution: the temperature axis is not free.** $T$ also changes $\mu_{max}$, $k_{cat}$ and $G$, so
> one stage-1 posterior cannot be reused across induction temperatures. There are two options. M1 can
> be fitted separately for each temperature arm, at the marginal cost of the OD series only, since
> those gels are being run anyway. Alternatively, explicit Arrhenius terms for those three parameters
> can be added inside M1, which then has 16 fitted parameters instead of 13. The first option is the
> less expensive. Every temperature arm below assumes its own stage-1 fit.

### 5.1 · On modelling chaperones

The obvious mechanistic extension is DnaK/GroEL titration: as the flux rises, free chaperone falls,
$k_f$ drops, and the collapse sharpens. This adds three parameters, none of which soluble-fraction
data can resolve separately. A free aggregation order $n$ absorbs the same sharpening with a single
parameter. $n$ is therefore fitted phenomenologically and should be reported as a sharpness
parameter, not as a measured aggregation stoichiometry; the model does not claim to describe
chaperone titration.

---
## 6 · Data: published measurements in place of the swap point

The swap-point cell reads the published measurements for arenin expressed in BL21(DE3) (doi 10.1186/s40643-025-00986-2, Table 1): soluble and insoluble target protein in mg/L, with replicate standard deviations, at 37 and 30 °C, 8 h after induction with 0.5 mM IPTG at OD₆₀₀ ≈ 0.6. The model uses the soluble fraction. The published standard deviations fix the concentration of the Beta likelihood, so no noise parameter has to be fitted to two data points.

In [ ]:
# ===================== LITERATURE DATA (not synthetic) =====================
# Hidalgo-Vazquez et al. 2025, Bioresour Bioprocess, doi 10.1186/s40643-025-00986-2,
# Table 1: arenin expressed in BL21(DE3), 0.5 mM IPTG at OD600 ~ 0.6, harvested 8 h
# later, soluble and insoluble target protein quantified in mg/L with replicate SD.
USE_SYNTHETIC = False
LIT = pd.read_csv("../../data/litdata/lit_M2_partition_gels.csv").set_index("temp_C")

TEMPS_C     = np.array([37.0, 30.0])      # the two induction temperatures measured
HARVEST_MIN = 480.0                       # 8 h post-induction
T_OBS_MIN   = np.array([HARVEST_MIN])
DOSES       = np.array([5500.])           # one induction condition; label only

P_OBS = {T: float(LIT.band_soluble[T]/(LIT.band_soluble[T]+LIT.band_pellet[T])) for T in TEMPS_C}
P_SD  = {T: float(np.sqrt((LIT.band_pellet[T]*LIT.band_soluble_sd[T])**2
                        + (LIT.band_soluble[T]*LIT.band_pellet_sd[T])**2)
                 /(LIT.band_soluble[T]+LIT.band_pellet[T])**2) for T in TEMPS_C}
# beta concentration implied by the published SD -- the likelihood uses the
# measured precision instead of fitting a noise parameter to two points.
NU_OBS = {T: P_OBS[T]*(1-P_OBS[T])/P_SD[T]**2 - 1.0 for T in TEMPS_C}

# Absolute synthesis flux of the published experiment is unknown. The M1 flux
# trajectory supplies the shape; its scale is anchored by the measured ratio of
# total target protein at harvest. s50_ref is therefore conditional on that scale,
# while dE/R -- which depends only on the ratio -- is not.
FLUX_RATIO = float(LIT.total_target_in_lysate_mg_L[37.0]/LIT.total_target_in_lysate_mg_L[30.0])
FLUX_SCALE = {37.0: 1.0, 30.0: 1.0/FLUX_RATIO}

THETA_BY_T = {T: dict(TRUE) for T in TEMPS_C}
_TGRID = np.linspace(0, T_END, 1201)
_SIM0  = simulate_m04(TRUE, DOSES, _TGRID)
SIM_BY_T = {}
for _T in TEMPS_C:
    _s = dict(_SIM0); _s["flux_uM_min"] = _SIM0["flux_uM_min"]*FLUX_SCALE[_T]
    SIM_BY_T[_T] = _s

def predict_grid(v, free_n=True):
    s50, dE = np.exp(v[0]), v[1]; n = v[2] if free_n else 2.0
    out = []
    for T in TEMPS_C:
        ls, lp = partition_quadrature(SIM_BY_T[T], THETA_BY_T[T], s50, dE, 273.15+T, n)
        out.append(ls/np.maximum(ls+lp, 1e-30))
    return np.stack(out)

_ti = int(np.argmin(np.abs(_TGRID - HARVEST_MIN)))
DATA = dict(dose_idx=np.zeros(2, int), t_idx=np.full(2, _ti), T_idx=np.array([0, 1]),
            y=np.array([P_OBS[37.0], P_OBS[30.0]]),
            nu=np.array([NU_OBS[37.0], NU_OBS[30.0]]))
df = pd.DataFrame(dict(lactose_uM=DOSES[0], time_min=HARVEST_MIN, temp_C=[37.0, 30.0],
                       soluble_fraction=DATA["y"], rep=0))
print("Measured data (arenin, doi 10.1186/s40643-025-00986-2 Table 1)")
for T in TEMPS_C:
    print(f"  {T:.0f} C : soluble fraction {P_OBS[T]:.3f} +/- {P_SD[T]:.3f}   "
          f"(beta concentration nu = {NU_OBS[T]:.0f})")
print(f"  measured flux ratio 37/30 C : {FLUX_RATIO:.2f}x")
df

---
## 7 · Calibration

The free parameters are $\log s_{50}^{ref}$ and $\Delta E/R$. The aggregation order is fixed at $n = 2$, because two measurements cannot identify it, and the Beta concentration $\nu$ is set by the published replicate SD instead of being fitted. $\varepsilon = 1$ is fixed (§4.5). The M1 parameters are held at the stage-1 posterior.

A self-contained adaptive-tempering SMC is used so that the notebook runs without PyMC.

In [ ]:
PARAMS = ["log_s50_ref", "dE_over_R", "n", "log_nu"]
PRIOR  = {"log_s50_ref": (np.log(60.), 1.2), "dE_over_R": (10000., 6000.),
          "n": (2.5, 1.0), "log_nu": (np.log(50.), 0.8)}
N_BOUNDS = (2.0, 6.0)

def log_prior(v, free_n=True):
    lp = 0.0
    for i, k in enumerate(PARAMS):
        if k == "n":
            if not free_n: continue
            if not (N_BOUNDS[0] <= v[2] <= N_BOUNDS[1]): return -np.inf
        m, sd = PRIOR[k]; lp += -0.5*((v[i]-m)/sd)**2
    return lp if v[1] > -5000.0 else -np.inf

def log_lik(v, data=None, free_n=True):
    # no bare try/except here: swallowing exceptions turns a code bug into a
    # silently degenerate likelihood, which is exactly how a broken sampler
    # produces a confident wrong posterior.
    data = DATA if data is None else data
    P = predict_grid(v, free_n=free_n)
    p = np.clip(P[data["T_idx"], data["dose_idx"], data["t_idx"]], 1e-6, 1-1e-6)
    # with two measurements the noise level cannot also be fitted: use the beta
    # concentration implied by the published replicate SD.
    nu = data["nu"] if "nu" in data else np.exp(v[3])
    return float(np.sum(beta_loglik(data["y"], p, nu)))

def log_post(v, data=None, free_n=True):
    lp = log_prior(v, free_n)
    return -np.inf if not np.isfinite(lp) else lp + log_lik(v, data, free_n)

In [ ]:
def smc(data=None, n_particles=400, free_n=True, ess_target=0.6, n_mh=8, rng=RNG):
    d = len(PARAMS)
    part = np.column_stack([rng.normal(*PRIOR[k], n_particles) for k in PARAMS])
    part[:, 2] = np.clip(part[:, 2], *N_BOUNDS)
    if not free_n: part[:, 2] = 2.0
    ll = np.array([log_lik(p, data, free_n) for p in part])
    beta_t, logZ = 0.0, 0.0
    while beta_t < 1.0:
        lo, hi = beta_t, 1.0
        for _ in range(50):
            b = 0.5*(lo+hi); w = (b-beta_t)*ll; w -= w.max(); W = np.exp(w)
            if W.sum()**2/np.sum(W**2) < ess_target*n_particles: hi = b
            else: lo = b
        b = min(1.0, max(lo, beta_t+1e-3))
        w = (b-beta_t)*ll; mx = w.max(); W = np.exp(w-mx)
        logZ += mx + np.log(W.mean()); W /= W.sum()
        idx = rng.choice(n_particles, n_particles, p=W)
        part, ll, beta_t = part[idx].copy(), ll[idx].copy(), b
        L = np.linalg.cholesky(np.cov(part.T) + 1e-10*np.eye(d))*(2.38/np.sqrt(d))
        for _ in range(n_mh):
            prop = part + rng.standard_normal((n_particles, d)) @ L.T
            if not free_n: prop[:, 2] = 2.0
            for j in range(n_particles):
                lpp = log_prior(prop[j], free_n)
                if not np.isfinite(lpp): continue
                llp = log_lik(prop[j], data, free_n)
                if np.log(rng.random()) < (lpp+beta_t*llp) - (log_prior(part[j],free_n)+beta_t*ll[j]):
                    part[j], ll[j] = prop[j], llp
        print(f"  beta = {beta_t:.4f}", end="\r")
    return part, logZ

import time; _t0 = time.time()
particles, logZ_free = smc(free_n=False)   # n fixed at 2: two points cannot identify it
print(f"\nSMC done in {time.time()-_t0:.0f} s | log Z = {logZ_free:.2f}")

---
## 8 · Recovery check

In [ ]:
post = pd.DataFrame({"s50_ref": np.exp(particles[:,0]), "dE_over_R": particles[:,1]})
ref = {"s50_ref": 50.0, "dE_over_R": 12000.0}          # values used in the synthetic run
rows = []
for k, v in ref.items():
    lo, hi = np.percentile(post[k], [3, 97])
    rows.append(dict(param=k, synthetic_run_value=v, posterior_median=np.median(post[k]),
                     lo94=lo, hi94=hi, synthetic_value_inside=bool(lo <= v <= hi)))
rec = pd.DataFrame(rows); display(rec.round(3))
print("\nTwo measurements, two free parameters: the posterior is a ridge, not a peak.")
print(f"posterior correlation s50_ref vs dE/R : {post.corr().iloc[0,1]:+.3f}")
print("\nClosed-form check (ratio inversion, independent of the absolute flux scale):")
_phi = lambda sig: 2*(np.sqrt(1+sig)-1)/sig
_sig = {T: brentq(lambda s: _phi(s)-P_OBS[T], 1e-9, 1e9) for T in TEMPS_C}
_ratio = (_sig[30.0]*FLUX_RATIO)/_sig[37.0]
_dE = np.log(_ratio)/(1/310.15 - 1/303.15)
print(f"  s50(37)/s50(30) = {_ratio:.3f}  ->  dE/R = {_dE:,.0f} K")
print(f"  SMC posterior median dE/R        = {np.median(post.dE_over_R):,.0f} K")
print("\nFlux change alone, with s50 temperature-independent:")
_p37_flux_only = _phi(_sig[30.0]*FLUX_RATIO)
print(f"  predicted 37 C soluble fraction = {_p37_flux_only:.3f}, measured {P_OBS[37.0]:.3f}"
      f"  ({(P_OBS[37.0]-_p37_flux_only)/P_SD[37.0]:+.1f} measurement SD)")

> **Caution: recovery is not validation.** The reference values in the table above are those of the synthetic run in notebook 03, so the table is informative only as a sampler check, together with the closed-form ratio inversion that follows it. Two measurements and two free parameters leave a ridge in the posterior and no data to hold out. Neither check can detect a wrong equation; the forward prediction in §10 is the test that can.

---
## 9 · Posterior predictive

In [ ]:
sel = particles[RNG.choice(len(particles), 60, replace=False)]
PP = np.stack([predict_grid(v) for v in sel])                 # (draws, T, dose, t)
tg = np.linspace(0, T_END, 1201)
fig, axs = plt.subplots(1, len(TEMPS_C), figsize=(4.2*len(TEMPS_C), 3.3), sharey=True)
for a, (k, T_C) in zip(np.atleast_1d(axs), enumerate(TEMPS_C)):
    for i, D in enumerate(DOSES):
        lo, md_, hi = np.percentile(PP[:, k, i, :], [2.5, 50, 97.5], axis=0)
        a.plot(tg/60, md_, lw=1.3, color=f"C{i}", label=f"{D:.0f} $\\mu$M")
        a.fill_between(tg/60, lo, hi, color=f"C{i}", alpha=.15, lw=0)
        m = (df.temp_C == T_C) & (df.lactose_uM == D)
        a.plot(df.time_min[m]/60, df.soluble_fraction[m], "o", ms=3.5, color=f"C{i}")
    a.set(title=f"{T_C:.0f} $^\\circ$C", xlabel="time (h)", ylim=(0, 1.05))
np.atleast_1d(axs)[0].set_ylabel("soluble fraction"); np.atleast_1d(axs)[0].legend(fontsize=6)
fig.tight_layout(); plt.show()

---
## 10 · Forward prediction: the falsification test

Two temperatures and two free parameters leave nothing to hold out. The calibration instead yields a forward prediction at temperatures that were not measured (25 and 18 °C), with no free parameter, using the synthesis flux of the 30 °C arm. A single gel at either temperature would test the Arrhenius closure.

In [ ]:
# Two temperatures and two free parameters leave nothing to hold out. What the
# calibration yields instead is a forward prediction at temperatures that were not measured.
PRED_T = np.array([25.0, 18.0])
pred = {}
for T in PRED_T:
    vals = []
    for v in particles:
        s50T = np.exp(v[0])*np.exp(v[1]*(1/(T+273.15) - 1/310.15))
        ls, lp = partition_quadrature(SIM_BY_T[30.0], THETA_BY_T[30.0], np.exp(v[0]), v[1],
                                      273.15+T, 2.0)
        vals.append(float((ls/np.maximum(ls+lp, 1e-30))[0, _ti]))
    pred[T] = np.array(vals)
print("Forward prediction at the 30 C synthesis flux (nothing free):")
for T in PRED_T:
    q = np.percentile(pred[T], [2.5, 50, 97.5])
    print(f"  {T:.0f} C : soluble fraction {q[1]:.3f}  95% [{q[0]:.3f}, {q[2]:.3f}]")
print("\nOne gel at either temperature falsifies the Arrhenius closure of Eq. (3.12).")

In [ ]:
fig, ax = plt.subplots(figsize=(5.8, 3.7))
v_med = np.median(particles, axis=0)
Tg = np.linspace(16, 39, 48)
curve = []
for T in Tg:
    ls, lp = partition_quadrature(SIM_BY_T[30.0], THETA_BY_T[30.0], np.exp(v_med[0]), v_med[1], 273.15+T, 2.0)
    curve.append(float((ls/np.maximum(ls+lp, 1e-30))[0, _ti]))
ax.plot(Tg, curve, color=BLUE, lw=2, label="model at the 30 °C flux")
# model evaluated at each measured condition, i.e. with that condition's own flux
fit_pts = [float(predict_grid(v_med, free_n=False)[k, 0, _ti]) for k, _ in enumerate(TEMPS_C)]
ax.plot(TEMPS_C, fit_pts, "D", color=SKY, ms=8, mec=INK, mew=.6, zorder=4,
        label="model at each measured flux (fit)")
for T in PRED_T:
    q = np.percentile(pred[T], [2.5, 50, 97.5])
    ax.errorbar([T], [q[1]], yerr=[[q[1]-q[0]], [q[2]-q[1]]], fmt="o", color=MID, ms=6,
                capsize=3, label="prediction, untested" if T == PRED_T[0] else None)
ax.errorbar(TEMPS_C, [P_OBS[T] for T in TEMPS_C], yerr=[P_SD[T] for T in TEMPS_C],
            fmt="o", color=NAVY, ms=7, capsize=3, zorder=5, label="measured (arenin)")
ax.set(xlabel="induction temperature (°C)", ylabel="soluble fraction", ylim=(0, 1.05),
       title="M2 calibrated on measured data")
ax.legend(fontsize=6.6, loc="lower left"); fig.tight_layout(); plt.show()
print("model minus measured at the two calibration points:",
      ", ".join(f"{T:.0f} C {fit_pts[k]-P_OBS[T]:+.3f}" for k, T in enumerate(TEMPS_C)))

---
## 11 · Export for M3

In [ ]:
best = np.median(particles, axis=0)
b = bridge(TRUE, DOSES, np.exp(best[0]), best[1], T_K=273.15+TEMPS_C[0], n=best[2])
exp = pd.DataFrame({"lactose_uM": DOSES,
                    "P_total_uM": b["P_M1"][:, -1],
                    "Sol_uM": b["Sol"][:, -1],
                    "Agg_uM": b["Agg"][:, -1],
                    "dimer_uM": b["dimer_uM"][:, -1],
                    "soluble_fraction_18h": b["gel_soluble_fraction"][:, -1]})
display(exp.round(1))
np.savez_compressed("M2_export_literature.npz",
                    particles=particles, param_names=np.array(PARAMS),
                    doses=DOSES, temps_C=TEMPS_C, t_grid=b["t"],
                    dimer_uM=b["dimer_uM"], Sol_uM=b["Sol"], Agg_uM=b["Agg"],
                    flux_nM_s=b["flux_nM_s"])
print("wrote M2_export_literature.npz  |  M3 consumes dimer_uM, not P_total_uM.")
print("Compare dimer_uM against c* from the Zlotnick module (notebook 04) before designing")
print("any experiment intended to measure c*: if c* is uM-scale, every arm sits far")
print("above it and assembly is not concentration-limited.")

---
## 12 · Next steps

**Interpretation of the calibration.** Two measurements constrain two parameters, so the posterior is a ridge (§8). The activation-energy parameter $\Delta E/R$ depends only on the ratio between the two measured conditions and not on the absolute flux scale. $s_{50}^{ref}$ is conditional on the assumed flux scale, which is anchored through the measured ratio of total target protein at harvest, and it should be quoted with that qualification.

**Test of the model.** A single gel at 25 °C or 18 °C, with soluble and insoluble target protein quantified in mg/L as in the source study, would test the Arrhenius closure against the forward prediction of §10.

**Protocol requirements** (see notebook 03, §3 and §12): OD-matched loading, whole-cell lysate for the stage-1 band, a clearing spin of at most 20 000 g, and a separate M1 fit for each induction temperature.
---
## 13 · References

1. Kiefhaber T, Rudolph R, Kohler HH, Buchner J. Protein aggregation in vitro and in vivo: a quantitative model of the kinetic competition between folding and aggregation. *Bio/Technology.* 1991;9(9):825–829.
2. Baneyx F, Mujacic M. Recombinant protein folding and misfolding in *Escherichia coli*. *Nat Biotechnol.* 2004;22(11):1399–1408.
3. Ventura S, Villaverde A. Protein quality in bacterial inclusion bodies. *Trends Biotechnol.* 2006;24(4):179–185.
4. Vera A, González-Montalbán N, Arís A, Villaverde A. The conformational quality of insoluble recombinant proteins is enhanced at low growth temperatures. *Biotechnol Bioeng.* 2007;96(6):1101–1106.
5. Schrödel A, de Marco A. Characterization of the aggregates formed during recombinant protein expression in bacteria. *BMC Biochem.* 2005;6:10.
6. Zlotnick A. To build a virus capsid: an equilibrium model of the self assembly of polyhedral protein complexes. *J Mol Biol.* 1994;241(1):59–67.
7. Plummer M. Cuts in Bayesian graphical models. *Stat Comput.* 2015;25(1):37–43.
8. Del Moral P, Doucet A, Jasra A. Sequential Monte Carlo samplers. *J R Stat Soc B.* 2006;68(3):411–436.
9. Raue A, Kreutz C, Maiwald T, et al. Structural and practical identifiability analysis of partially observed dynamical models. *Bioinformatics.* 2009;25(15):1923–1929.
10. Gelman A, Hill J. *Data Analysis Using Regression and Multilevel/Hierarchical Models.* CUP; 2007.

Carried over from the M1 notebooks: Studier 2005; Scott 2010; Klumpp 2009; Rocke & Durbin 2001;
Kennedy & O'Hagan 2001.